# 🛍️ Customer Segmentation — Part 2: RFM Analysis

**Project:** Customer Segmentation using RFM Analysis & K-Means Clustering  
**Input:** `data/transactions_clean.parquet` (from notebook 01)  
**Output:** `data/rfm_scores.parquet` — one row per customer with RFM scores  

---

## What is RFM?

RFM is a proven marketing framework that scores every customer on three dimensions:

| Dimension | Question | High score means... |
|-----------|----------|---------------------|
| **R**ecency | How recently did they buy? | Bought very recently |
| **F**requency | How often do they buy? | Buys very often |
| **M**onetary | How much do they spend? | Spends a lot |

## What we'll cover
1. Compute raw RFM values per customer
2. Score each dimension (1–5 scale)
3. Visualize RFM distributions
4. Analyze RFM correlations
5. Manual segment labels (rule-based, before clustering)
6. Save RFM table for clustering notebook


## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import timedelta
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.titleweight': 'bold',
})

PALETTE  = ['#1D9E75', '#378ADD', '#D85A30', '#7F77DD', '#BA7517']
PRIMARY  = '#1D9E75'

print('Libraries loaded ✓')

## 1. Load Clean Data

In [ ]:
df = pd.read_parquet('../data/transactions_clean.parquet')

print(f'Transactions : {len(df):,}')
print(f'Customers    : {df["Customer ID"].nunique():,}')
print(f'Date range   : {df["InvoiceDate"].min().date()} → {df["InvoiceDate"].max().date()}')
df.head(3)

## 2. Compute Raw RFM Values

We set the **snapshot date** as 1 day after the last transaction.  
This is the reference point for calculating Recency.

In [ ]:
# Snapshot date = 1 day after last transaction in dataset
snapshot_date = df['InvoiceDate'].max() + timedelta(days=1)
print(f'Snapshot date: {snapshot_date.date()}')

# ── Compute RFM per customer ─────────────────────────────────────────────────
rfm = (
    df.groupby('Customer ID')
    .agg(
        Recency   = ('InvoiceDate',   lambda x: (snapshot_date - x.max()).days),
        Frequency = ('Invoice',        'nunique'),
        Monetary  = ('TotalRevenue',   'sum')
    )
    .reset_index()
)

print(f'\nRFM table shape: {rfm.shape}')
rfm.head(10)

In [ ]:
# Descriptive statistics
rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2)

### Interpretation of raw values
- **Recency**: Days since last purchase — lower is better (bought recently)
- **Frequency**: Number of unique orders — higher is better
- **Monetary**: Total spend in GBP — higher is better

> Note: Monetary values are right-skewed — a few customers spend dramatically more than the median. We'll handle this when normalizing for clustering.

## 3. Score Each RFM Dimension (1–5 Scale)

We use **quintile-based scoring**:
- Split each dimension into 5 equal groups
- Assign scores 1–5 where **5 is always best**
- For Recency: low days = high score (inverted)
- For Frequency & Monetary: high value = high score

In [ ]:
# ── Quintile scoring ─────────────────────────────────────────────────────────
rfm['R_Score'] = pd.qcut(rfm['Recency'],   q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'),  q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# ── Composite RFM score ──────────────────────────────────────────────────────
rfm['RFM_Score']  = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']
rfm['RFM_Label']  = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

print('Score distribution:')
print(rfm[['R_Score','F_Score','M_Score','RFM_Score']].describe().round(2))

In [ ]:
rfm.head(10)

## 4. Visualize RFM Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('RFM Raw Values & Score Distributions', fontsize=16, fontweight='bold')

colors = [PRIMARY, '#378ADD', '#7F77DD']

# Row 1: Raw values (capped at 95th percentile for readability)
for i, (col, label, color) in enumerate(zip(
    ['Recency', 'Frequency', 'Monetary'],
    ['Days since last purchase', 'Number of orders', 'Total spend (GBP)'],
    colors
)):
    ax = axes[0, i]
    cap = rfm[col].quantile(0.95)
    data = rfm[col].clip(upper=cap)
    ax.hist(data, bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'Raw {col}')
    ax.set_xlabel(label)
    ax.set_ylabel('Customers')
    if col == 'Monetary':
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: f'£{x:,.0f}'))
    median_val = rfm[col].median()
    ax.axvline(median_val, color='red', linestyle='--', alpha=0.7, linewidth=1.5)
    ax.text(median_val * 1.05, ax.get_ylim()[1] * 0.9,
            f'Median\n{median_val:.0f}', fontsize=9, color='red')

# Row 2: Score distributions (1–5)
for i, (col, color) in enumerate(zip(
    ['R_Score', 'F_Score', 'M_Score'],
    colors
)):
    ax = axes[1, i]
    score_counts = rfm[col].value_counts().sort_index()
    bars = ax.bar(score_counts.index, score_counts.values,
                  color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel('Score (1=worst, 5=best)')
    ax.set_ylabel('Customers')
    ax.set_xticks([1, 2, 3, 4, 5])
    # Add count labels on bars
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 10,
                f'{int(h):,}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/figures/rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── RFM Score heatmap (R vs F, colored by avg Monetary) ─────────────────────
rfm_pivot = (
    rfm.groupby(['R_Score', 'F_Score'])['Monetary']
    .mean()
    .reset_index()
    .pivot(index='R_Score', columns='F_Score', values='Monetary')
    .sort_index(ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    rfm_pivot,
    annot=True,
    fmt='.0f',
    cmap='YlGn',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Avg Monetary Value (GBP)', 'shrink': 0.8}
)
ax.set_title('Avg Monetary Value by Recency × Frequency Score', fontweight='bold')
ax.set_xlabel('Frequency Score →  (1=low, 5=high)')
ax.set_ylabel('Recency Score →  (1=old, 5=recent)')
plt.tight_layout()
plt.savefig('../outputs/figures/rfm_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Total RFM Score distribution ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

score_dist = rfm['RFM_Score'].value_counts().sort_index()

# Color by score: red (low) → green (high)
cmap = plt.cm.RdYlGn
norm_scores = (score_dist.index - score_dist.index.min()) / \
              (score_dist.index.max() - score_dist.index.min())
bar_colors = [cmap(v) for v in norm_scores]

bars = ax.bar(score_dist.index, score_dist.values, color=bar_colors, edgecolor='white')
ax.set_title('Distribution of Total RFM Scores (3–15)', fontweight='bold')
ax.set_xlabel('RFM Score (3=worst, 15=best)')
ax.set_ylabel('Number of Customers')
ax.set_xticks(range(3, 16))

# Annotate zones
ax.axvspan(2.5,  6.5, alpha=0.05, color='red',   label='At-risk / Lost')
ax.axvspan(6.5, 10.5, alpha=0.05, color='orange', label='Mid-tier')
ax.axvspan(10.5, 15.5,alpha=0.05, color='green',  label='High-value')

ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/figures/rfm_total_score.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. RFM Correlation Analysis

In [ ]:
# ── Scatter matrix: R, F, M scores ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('RFM Score Relationships', fontsize=14, fontweight='bold')

pairs = [('R_Score', 'F_Score'), ('R_Score', 'M_Score'), ('F_Score', 'M_Score')]

for ax, (x, y) in zip(axes, pairs):
    # Jitter to show density
    jitter = 0.2
    xj = rfm[x] + np.random.uniform(-jitter, jitter, len(rfm))
    yj = rfm[y] + np.random.uniform(-jitter, jitter, len(rfm))
    ax.scatter(xj, yj, alpha=0.15, s=8, color=PRIMARY)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f'{x} vs {y}')
    ax.set_xticks([1,2,3,4,5])
    ax.set_yticks([1,2,3,4,5])
    corr = rfm[[x, y]].corr().iloc[0, 1]
    ax.text(0.05, 0.92, f'r = {corr:.2f}', transform=ax.transAxes,
            fontsize=10, color='#444')

plt.tight_layout()
plt.savefig('../outputs/figures/rfm_scatter_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Rule-Based Segment Labels

Before we cluster, let's apply **industry-standard rule-based labels**.  
These give us a baseline — we'll compare them against K-Means results in notebook 03.

Segments based on RFM score combinations:

In [ ]:
def rfm_segment_label(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    score   = row['RFM_Score']

    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r >= 3 and f >= 2 and m >= 2:
        return 'Potential Loyalists'
    elif r == 3 and f <= 2:
        return 'Promising'
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r <= 2 and f >= 4 and m >= 4:
        return 'Cannot Lose Them'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost'
    else:
        return 'Hibernating'

rfm['Segment_RuleBased'] = rfm.apply(rfm_segment_label, axis=1)

seg_counts = rfm['Segment_RuleBased'].value_counts()
print('Segment sizes:')
print(seg_counts.to_string())

In [ ]:
# ── Segment visualization ────────────────────────────────────────────────────
seg_stats = (
    rfm.groupby('Segment_RuleBased')
    .agg(
        Customers    = ('Customer ID', 'count'),
        Avg_Recency  = ('Recency',    'mean'),
        Avg_Frequency= ('Frequency',  'mean'),
        Avg_Monetary = ('Monetary',   'mean'),
        Total_Revenue= ('Monetary',   'sum'),
    )
    .round(1)
    .sort_values('Total_Revenue', ascending=False)
)

seg_stats['Revenue_Share_%'] = (seg_stats['Total_Revenue'] /
                                 seg_stats['Total_Revenue'].sum() * 100).round(1)
print(seg_stats.to_string())

In [ ]:
# ── Visual: segment size vs revenue ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Rule-Based Customer Segments', fontsize=15, fontweight='bold')

seg_colors = {
    'Champions':          '#1D9E75',
    'Loyal Customers':    '#378ADD',
    'Potential Loyalists':'#7F77DD',
    'New Customers':      '#5DCAA5',
    'Promising':          '#85B7EB',
    'At Risk':            '#BA7517',
    'Cannot Lose Them':   '#D85A30',
    'Hibernating':        '#888780',
    'Lost':               '#B4B2A9',
}

ordered_segs = seg_stats.index.tolist()
plot_colors  = [seg_colors.get(s, '#888') for s in ordered_segs]

# Plot 1: Customer count
ax = axes[0]
bars = ax.barh(ordered_segs[::-1], seg_stats.loc[ordered_segs[::-1], 'Customers'],
               color=plot_colors[::-1], edgecolor='white')
ax.set_title('Customers per Segment')
ax.set_xlabel('Number of Customers')
for bar in bars:
    w = bar.get_width()
    ax.text(w + 5, bar.get_y() + bar.get_height()/2,
            f'{int(w):,}', va='center', fontsize=9)

# Plot 2: Revenue share
ax = axes[1]
wedges, texts, autotexts = ax.pie(
    seg_stats.loc[ordered_segs, 'Total_Revenue'],
    labels=None,
    colors=plot_colors,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.75,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for at in autotexts:
    at.set_fontsize(9)
ax.set_title('Revenue Share by Segment')
legend_patches = [mpatches.Patch(color=seg_colors.get(s,'#888'), label=s)
                  for s in ordered_segs]
ax.legend(handles=legend_patches, loc='lower left',
          bbox_to_anchor=(-0.3, -0.15), fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig('../outputs/figures/segments_rulebased.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Snake / radar-like bar chart: avg R, F, M per segment ────────────────────
seg_rfm_profile = (
    rfm.groupby('Segment_RuleBased')[['R_Score','F_Score','M_Score']]
    .mean()
    .round(2)
    .loc[ordered_segs]
)

fig, ax = plt.subplots(figsize=(12, 6))
x      = np.arange(len(ordered_segs))
width  = 0.25

b1 = ax.bar(x - width, seg_rfm_profile['R_Score'], width, label='Recency',   color=PRIMARY,    alpha=0.85)
b2 = ax.bar(x,         seg_rfm_profile['F_Score'], width, label='Frequency', color='#378ADD',  alpha=0.85)
b3 = ax.bar(x + width, seg_rfm_profile['M_Score'], width, label='Monetary',  color='#7F77DD',  alpha=0.85)

ax.set_title('Average R, F, M Scores by Segment', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(ordered_segs, rotation=30, ha='right')
ax.set_ylabel('Average Score (1–5)')
ax.set_ylim(0, 6)
ax.legend()
ax.axhline(3, color='gray', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('../outputs/figures/rfm_segment_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Business Insights from RFM

Let's quantify the business opportunity per segment.

In [ ]:
total_revenue = rfm['Monetary'].sum()

print('='*65)
print('BUSINESS OPPORTUNITY SUMMARY')
print('='*65)

for seg in ordered_segs:
    subset    = rfm[rfm['Segment_RuleBased'] == seg]
    n         = len(subset)
    rev       = subset['Monetary'].sum()
    rev_share = rev / total_revenue * 100
    avg_rev   = subset['Monetary'].mean()
    avg_rec   = subset['Recency'].mean()
    print(f"\n{seg}")
    print(f"  Customers     : {n:,}")
    print(f"  Total revenue : £{rev:,.0f}  ({rev_share:.1f}% of total)")
    print(f"  Avg revenue   : £{avg_rev:,.0f}")
    print(f"  Avg recency   : {avg_rec:.0f} days")

## 8. Save RFM Table

In [ ]:
import os
os.makedirs('../outputs/figures', exist_ok=True)

rfm.to_parquet('../data/rfm_scores.parquet', index=False)
rfm.to_csv('../outputs/rfm_scores.csv', index=False)

print(f'Saved rfm_scores.parquet & rfm_scores.csv')
print(f'Columns: {rfm.columns.tolist()}')
print(f'Shape  : {rfm.shape}')
rfm.head()

## ✅ RFM Analysis Summary

| Output | Detail |
|--------|--------|
| **Customers scored** | All customers with R, F, M scores (1–5) |
| **Composite score** | RFM_Score (3–15), higher = better |
| **Rule-based segments** | 9 segments with business labels |
| **Top segment** | Champions — highest R, F, M across the board |
| **Biggest risk** | At Risk + Lost customers hold significant past revenue |

### Key findings
- **Champions** are few but generate a disproportionate share of revenue
- **At Risk** and **Cannot Lose Them** need immediate win-back campaigns
- **New Customers** have high Recency but low Frequency — nurture them now
- **Lost** customers have very low scores across all dimensions — minimal ROI to target

### Limitation of rule-based approach
> Rules are manually defined and may not reflect natural customer groupings in the data.  
> **➡️ Next: Notebook 03 — K-Means Clustering** to let the data define the segments objectively.